# Machine Learning-Based Prediction of 30-Day Hospital Readmission in Diabetic Patients

## Import Libraries

In [1]:
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

RANDOM_STATE = 42


## Load Data and Define the Target

In [2]:
df = pd.read_csv("../data/3. processed/final_dataset.csv")

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())


Dataset shape: (101766, 28)
Columns: ['patient_nbr', 'race', 'gender', 'age', 'admission_type', 'discharge_disposition', 'admission_source', 'time_in_hospital', 'payer_code', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'A1Cresult', 'change', 'diabetesMed', 'readmitted_binary', 'total_diabetes_drugs', 'insulin_flag', 'oral_med_flag', 'high_medication_count_10plus']


## Separate the Target, Patient Groups, and Predictors

In [3]:
# patient_nbr is used only to keep encounters from the same patient in a single data partition. It must not be a model predictor.

if "patient_nbr" not in df.columns:
    raise ValueError(
        "patient_nbr is missing from final_dataset.csv. "
        "Retain it in the Feature Engineering notebook for grouped splitting."
    )

if "readmitted_binary" not in df.columns:
    raise ValueError("readmitted_binary is missing from final_dataset.csv.")

groups = df["patient_nbr"].copy()
y = df["readmitted_binary"].astype("int8")

# Exclude the grouping ID and target. Also exclude the original readmitted label if it remains in the file, because it would leak the target.

columns_to_exclude = ["patient_nbr", "readmitted_binary"]
if "readmitted" in df.columns:
    columns_to_exclude.append("readmitted")

X = df.drop(columns=columns_to_exclude)

assert X.columns.is_unique
assert y.isin([0, 1]).all()
assert groups.notna().all()

print("Rows:", len(df))
print("Unique patients:", groups.nunique())
print("Positive-class rate:", round(y.mean(), 4))
print("Predictor count:", X.shape[1])


Rows: 101766
Unique patients: 71518
Positive-class rate: 0.1116
Predictor count: 26


## Split into Training, Validation, and Test Sets

In [4]:
# GroupShuffleSplit splits patient groups, not individual encounters.
# Because patients can have different numbers of encounters, row proportions may differ from 60/20/20. The same patient must never occur across partitions.

outer_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)
trainval_idx, test_idx = next(outer_split.split(X, y, groups))

X_trainval = X.iloc[trainval_idx].reset_index(drop=True)
y_trainval = y.iloc[trainval_idx].reset_index(drop=True)
g_trainval = groups.iloc[trainval_idx].reset_index(drop=True)

X_test = X.iloc[test_idx].reset_index(drop=True)
y_test = y.iloc[test_idx].reset_index(drop=True)
g_test = groups.iloc[test_idx].reset_index(drop=True)

# Split the remaining patient groups into training and validation partitions.
inner_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=RANDOM_STATE + 1
)
train_idx, val_idx = next(
    inner_split.split(X_trainval, y_trainval, g_trainval)
)

X_train = X_trainval.iloc[train_idx].reset_index(drop=True)
y_train = y_trainval.iloc[train_idx].reset_index(drop=True)
g_train = g_trainval.iloc[train_idx].reset_index(drop=True)

X_val = X_trainval.iloc[val_idx].reset_index(drop=True)
y_val = y_trainval.iloc[val_idx].reset_index(drop=True)
g_val = g_trainval.iloc[val_idx].reset_index(drop=True)

# Confirm that patient groups do not overlap.
assert set(g_train).isdisjoint(set(g_val))
assert set(g_train).isdisjoint(set(g_test))
assert set(g_val).isdisjoint(set(g_test))

for split_name, labels, patient_ids in [
    ("train", y_train, g_train),
    ("validation", y_val, g_val),
    ("test", y_test, g_test)
]:
    print(
        f"{split_name:10s} "
        f"rows={len(labels):6d}, "
        f"patients={patient_ids.nunique():6d}, "
        f"positive_rate={labels.mean():.4f}"
    )


train      rows= 61215, patients= 42910, positive_rate=0.1133
validation rows= 20398, patients= 14304, positive_rate=0.1113
test       rows= 20153, patients= 14304, positive_rate=0.1067


## Define the Preprocessing Pipeline

In [5]:
# This project previously created is_polypharmacy in Feature Engineering.
# We use the alternative name high_medication_count_10plus.
# Use whichever of those names actually exists in final_dataset.csv.

polypharmacy_candidates = [
    "high_medication_count_10plus",
    "is_polypharmacy"
]
polypharmacy_feature = next(
    (col for col in polypharmacy_candidates if col in X.columns),
    None
)

if polypharmacy_feature is None:
    raise ValueError(
        "Neither 'is_polypharmacy' nor 'high_medication_count_10plus' "
        "was found in final_dataset.csv. Check the feature name in notebook 04."
    )

numeric_features = [
    "age",
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
    "total_diabetes_drugs",
    "insulin_flag",
    "oral_med_flag",
    "change",
    "diabetesMed",
    polypharmacy_feature
]

categorical_features = [
    "race",
    "gender",
    "payer_code",
    "A1Cresult",
    "admission_type",
    "discharge_disposition",
    "admission_source",
    "medical_specialty",
    "diag_1",
    "diag_2",
    "diag_3"
]

missing_features = sorted(
    (set(numeric_features) | set(categorical_features)) - set(X.columns)
)
if missing_features:
    raise ValueError(
        "Expected predictors are missing from final_dataset.csv: "
        f"{missing_features}"
    )

print("Polypharmacy feature used:", polypharmacy_feature)
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))


Polypharmacy feature used: high_medication_count_10plus
Numeric features: 15
Categorical features: 11


In [6]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    # Learn category frequencies on training data only.
    ("encoder", OneHotEncoder(
        handle_unknown="infrequent_if_exist",
        min_frequency=100
    ))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])


## Define Candidate Models

In [7]:
models = {
    "Dummy prior baseline": DummyClassifier(strategy="prior"),
    "Logistic Regression": LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=1
    ),
    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced",
        random_state=RANDOM_STATE,
        min_samples_leaf=10
    )
}


## Train Models and Compare on Validation Data

In [8]:
# Fit each candidate on training data only.
# Use validation data for model comparison; do not evaluate on the test set here.

fitted_models = {}
validation_rows = []

for name, estimator in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", clone(preprocessor)),
        ("classifier", estimator)
    ])

    pipeline.fit(X_train, y_train)
    fitted_models[name] = pipeline

    val_prob = pipeline.predict_proba(X_val)[:, 1]
    val_pred_05 = (val_prob >= 0.5).astype("int8")

    validation_rows.append({
        "Model": name,
        "Validation ROC-AUC": roc_auc_score(y_val, val_prob),
        "Validation Average Precision": average_precision_score(y_val, val_prob),
        "Precision @ 0.5": precision_score(
            y_val, val_pred_05, zero_division=0
        ),
        "Recall @ 0.5": recall_score(
            y_val, val_pred_05, zero_division=0
        ),
        "F1 @ 0.5": f1_score(
            y_val, val_pred_05, zero_division=0
        ),
        "Accuracy @ 0.5": accuracy_score(y_val, val_pred_05)
    })

validation_results = pd.DataFrame(validation_rows).sort_values(
    "Validation Average Precision",
    ascending=False
).reset_index(drop=True)

display(validation_results)


,Model,Validation ROC-AUC,Validation Average Precision,Precision @ 0.5,Recall @ 0.5,F1 @ 0.5,Accuracy @ 0.5
0,Logistic Regression,0.677485,0.225329,0.182512,0.569163,0.276393,0.668350
1,Random Forest,0.664583,0.215221,0.600000,0.001322,0.002637,0.888764
2,Decision Tree,0.575920,0.151668,0.145642,0.398238,0.213283,0.673056
3,Dummy prior baseline,0.500000,0.111285,0.000000,0.000000,0.000000,0.888715


### Model-selection rule

Choose the candidate with the highest **validation average precision**. Average precision summarizes precision-recall performance across thresholds and is useful for an imbalanced target. The dummy model is included as a baseline so you can see whether the trained models outperform a simple prevalence-based prediction.

## Select a Decision Threshold Using Validation Data

In [9]:
best_model_name = validation_results.loc[0, "Model"]
final_model = fitted_models[best_model_name]

val_prob = final_model.predict_proba(X_val)[:, 1]
threshold_grid = np.arange(0.05, 0.96, 0.01)
threshold_rows = []

for threshold in threshold_grid:
    val_pred = (val_prob >= threshold).astype("int8")
    threshold_rows.append({
        "threshold": threshold,
        "precision": precision_score(y_val, val_pred, zero_division=0),
        "recall": recall_score(y_val, val_pred, zero_division=0),
        "f1": f1_score(y_val, val_pred, zero_division=0)
    })

threshold_results = pd.DataFrame(threshold_rows)
best_row = threshold_results.loc[threshold_results["f1"].idxmax()]
best_threshold = float(best_row["threshold"])

print("Selected model:", best_model_name)
print("Threshold selected on validation set:", best_threshold)
display(best_row.to_frame(name="Selected validation result"))


Selected model: Logistic Regression
Threshold selected on validation set: 0.5600000000000002


,Selected validation result
threshold,0.560000
precision,0.214126
recall,0.420705
f1,0.283804


The threshold above is selected using **validation F1**, not test labels. F1 is a project-level selection criterion, not a clinical recommendation; real clinical use would need to account for the relative consequences of false negatives and false positives.

## Final Test Evaluation — Run Only After Model and Threshold Are Fixed

In [10]:

# Final test evaluation for the full cohort.
# Do not use test labels to select the model or threshold.

test_prob = final_model.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= best_threshold).astype("int8")

print("Cohort: Full dataset")
print("Selected model:", best_model_name)
print("Locked threshold:", best_threshold)

cm = confusion_matrix(y_test, test_pred, labels=[0, 1])

print("\nConfusion matrix (rows=true [0, 1], columns=predicted [0, 1]):")
print(cm)

print("\nClassification report:")
print(classification_report(
    y_test,
    test_pred,
    labels=[0, 1],
    zero_division=0
))

test_roc_auc = roc_auc_score(y_test, test_prob)
test_average_precision = average_precision_score(y_test, test_prob)

tn, fp, fn, tp = cm.ravel()
specificity = tn / (tn + fp) if (tn + fp) else np.nan

test_precision = precision_score(y_test, test_pred, zero_division=0)
test_recall = recall_score(y_test, test_pred, zero_division=0)
test_f1 = f1_score(y_test, test_pred, zero_division=0)
test_accuracy = accuracy_score(y_test, test_pred)

print("\nFull-cohort test metrics")
print("Test ROC-AUC:", round(test_roc_auc, 4))
print("Test average precision:", round(test_average_precision, 4))
print("Test precision:", round(test_precision, 4))
print("Test recall:", round(test_recall, 4))
print("Test F1:", round(test_f1, 4))
print("Test specificity:", round(specificity, 4))
print("Test accuracy:", round(test_accuracy, 4))

# Store these results before evaluating the second cohort.
# The validation average precision belongs to the selected model.
selected_validation_ap = validation_results.loc[
    validation_results["Model"] == best_model_name,
    "Validation Average Precision"
].iloc[0]

full_cohort_metrics = {
    "Cohort": "Full cohort",
    "Rows": len(df),
    "Positive rate": float(y.mean()),
    "Selected model": best_model_name,
    "Validation average precision": float(selected_validation_ap),
    "Threshold": float(best_threshold),
    "Test ROC-AUC": float(test_roc_auc),
    "Test average precision": float(test_average_precision),
    "Test precision": float(test_precision),
    "Test recall": float(test_recall),
    "Test F1": float(test_f1),
    "Test specificity": float(specificity),
    "Test accuracy": float(test_accuracy)
}

print("\nFull-cohort metrics stored for the cohort comparison.")


Cohort: Full dataset
Selected model: Logistic Regression
Locked threshold: 0.5600000000000002

Confusion matrix (rows=true [0, 1], columns=predicted [0, 1]):
[[14565  3437]
 [ 1308   843]]

Classification report:
              precision    recall  f1-score   support

           0       0.92      0.81      0.86     18002
           1       0.20      0.39      0.26      2151

    accuracy                           0.76     20153
   macro avg       0.56      0.60      0.56     20153
weighted avg       0.84      0.76      0.80     20153


Full-cohort test metrics
Test ROC-AUC: 0.6641
Test average precision: 0.2025
Test precision: 0.197
Test recall: 0.3919
Test F1: 0.2622
Test specificity: 0.8091
Test accuracy: 0.7646

Full-cohort metrics stored for the cohort comparison.


## Save Separate Research Artifacts

In [11]:
# Save separately so the current app's final_model.pkl is not overwritten.
joblib.dump(final_model, "../models/research_final_model.pkl")
joblib.dump(best_threshold, "../models/research_threshold.pkl")

print("Saved research model and validation-selected threshold.")


Saved research model and validation-selected threshold.


In [12]:

# Sensitivity analysis: eligible-discharge cohort
#
# This experiment excludes encounters whose discharge disposition
# indicates death or hospice. The full-cohort experiment above remains
# unchanged. Both experiments use patient-grouped splits and the same
# validation-based model-selection and threshold-selection procedures.

eligible_df = pd.read_csv(
    "../data/3. processed/eligible_discharge_dataset.csv"
)

print("Eligible-discharge dataset shape:", eligible_df.shape)

# Include the general 'Expired' label to handle the current dataset,
# as well as the more specific labels expected after the mapping fix.
terminal_or_hospice_dispositions = {
    "Expired",
    "Hospice / home",
    "Hospice / medical facility",
    "Expired at home. Medicaid only, hospice.",
    "Expired in a medical facility. Medicaid only, hospice.",
    "Expired, place unknown. Medicaid only, hospice."
}

remaining_excluded = eligible_df[
    eligible_df["discharge_disposition"].isin(
        terminal_or_hospice_dispositions
    )
]

if not remaining_excluded.empty:
    raise ValueError(
        "The eligible-discharge dataset still contains death/hospice "
        "disposition categories. Re-run Cell 57 in Notebook 04 and "
        "check the discharge-disposition labels."
    )

if "patient_nbr" not in eligible_df.columns:
    raise ValueError("patient_nbr is missing from the eligible dataset.")

if "readmitted_binary" not in eligible_df.columns:
    raise ValueError("readmitted_binary is missing from the eligible dataset.")

eligible_groups = eligible_df["patient_nbr"].copy()
eligible_y = eligible_df["readmitted_binary"].astype("int8")

eligible_columns_to_exclude = [
    "patient_nbr",
    "readmitted_binary"
]

if "readmitted" in eligible_df.columns:
    eligible_columns_to_exclude.append("readmitted")

eligible_X = eligible_df.drop(
    columns=eligible_columns_to_exclude
)

if eligible_X.columns.duplicated().any():
    raise ValueError("Duplicate predictor column names were found.")

if not eligible_y.isin([0, 1]).all():
    raise ValueError("The target must contain only 0 and 1.")

if eligible_groups.isna().any():
    raise ValueError("Some patient_nbr values are missing.")

# Confirm that the predictors required by the existing preprocessor exist.
required_features = set(numeric_features) | set(categorical_features)
missing_eligible_features = sorted(
    required_features - set(eligible_X.columns)
)

if missing_eligible_features:
    raise ValueError(
        "Required predictors are missing from the eligible dataset: "
        f"{missing_eligible_features}"
    )

print("Eligible rows:", len(eligible_df))
print("Eligible unique patients:", eligible_groups.nunique())
print("Eligible positive-class rate:", round(eligible_y.mean(), 4))
print("Eligible predictor count:", eligible_X.shape[1])


# ------------------------------------------------------------
# Reuse full-cohort patient assignments for eligible cohort
# ------------------------------------------------------------

# These patient IDs were assigned in the full-cohort split (Cell 8).
full_train_patient_ids = set(g_train.unique())
full_val_patient_ids = set(g_val.unique())
full_test_patient_ids = set(g_test.unique())

# Verify that every eligible patient exists in the full-cohort split.
full_split_patient_ids = (
    full_train_patient_ids
    | full_val_patient_ids
    | full_test_patient_ids
)

eligible_patient_ids = set(eligible_groups.unique())

unassigned_patients = (
    eligible_patient_ids - full_split_patient_ids
)

if unassigned_patients:
    raise ValueError(
        "Some eligible patients were not assigned "
        "to a full-cohort data partition."
    )

# Assign eligible encounters using their patient's
# existing full-cohort partition.
eligible_train_mask = eligible_groups.isin(
    full_train_patient_ids
)

eligible_val_mask = eligible_groups.isin(
    full_val_patient_ids
)

eligible_test_mask = eligible_groups.isin(
    full_test_patient_ids
)

# Ensure every eligible encounter belongs to exactly one partition.
assignment_count = (
    eligible_train_mask.astype(int)
    + eligible_val_mask.astype(int)
    + eligible_test_mask.astype(int)
)

assert (assignment_count == 1).all(), (
    "Some eligible encounters have missing or multiple assignments."
)

# Create training partition.
eligible_X_train = (
    eligible_X.loc[eligible_train_mask].reset_index(drop=True)
)
eligible_y_train = (
    eligible_y.loc[eligible_train_mask].reset_index(drop=True)
)
eligible_g_train = (
    eligible_groups.loc[eligible_train_mask].reset_index(drop=True)
)

# Create validation partition.
eligible_X_val = (
    eligible_X.loc[eligible_val_mask].reset_index(drop=True)
)
eligible_y_val = (
    eligible_y.loc[eligible_val_mask].reset_index(drop=True)
)
eligible_g_val = (
    eligible_groups.loc[eligible_val_mask].reset_index(drop=True)
)

# Create test partition.
eligible_X_test = (
    eligible_X.loc[eligible_test_mask].reset_index(drop=True)
)
eligible_y_test = (
    eligible_y.loc[eligible_test_mask].reset_index(drop=True)
)
eligible_g_test = (
    eligible_groups.loc[eligible_test_mask].reset_index(drop=True)
)

# Verify patient-level separation.
eligible_train_patients = set(eligible_g_train.unique())
eligible_val_patients = set(eligible_g_val.unique())
eligible_test_patients = set(eligible_g_test.unique())

assert eligible_train_patients.isdisjoint(eligible_val_patients)
assert eligible_train_patients.isdisjoint(eligible_test_patients)
assert eligible_val_patients.isdisjoint(eligible_test_patients)

# Verify that each eligible partition inherits its
# corresponding full-cohort patient assignment.
assert eligible_train_patients.issubset(full_train_patient_ids)
assert eligible_val_patients.issubset(full_val_patient_ids)
assert eligible_test_patients.issubset(full_test_patient_ids)

print("Eligible cohort uses the same patient assignments as the full cohort.")

for split_name, labels, patient_ids in [
    ("eligible train", eligible_y_train, eligible_g_train),
    ("eligible validation", eligible_y_val, eligible_g_val),
    ("eligible test", eligible_y_test, eligible_g_test)
]:
    print(
        f"{split_name:22s} "
        f"rows={len(labels):6d}, "
        f"patients={patient_ids.nunique():6d}, "
        f"positive_rate={labels.mean():.4f}"
    )


# ------------------------------------------------------------
# Train candidate models using training data only
# ------------------------------------------------------------

eligible_fitted_models = {}
eligible_validation_rows = []

for name, estimator in models.items():
    eligible_pipeline = Pipeline(steps=[
        ("preprocessor", clone(preprocessor)),
        ("classifier", clone(estimator))
    ])

    eligible_pipeline.fit(eligible_X_train, eligible_y_train)
    eligible_fitted_models[name] = eligible_pipeline

    eligible_val_prob = eligible_pipeline.predict_proba(
        eligible_X_val
    )[:, 1]

    eligible_val_pred_05 = (
        eligible_val_prob >= 0.5
    ).astype("int8")

    eligible_validation_rows.append({
        "Model": name,
        "Validation ROC-AUC": roc_auc_score(
            eligible_y_val, eligible_val_prob
        ),
        "Validation Average Precision": average_precision_score(
            eligible_y_val, eligible_val_prob
        ),
        "Precision @ 0.5": precision_score(
            eligible_y_val, eligible_val_pred_05, zero_division=0
        ),
        "Recall @ 0.5": recall_score(
            eligible_y_val, eligible_val_pred_05, zero_division=0
        ),
        "F1 @ 0.5": f1_score(
            eligible_y_val, eligible_val_pred_05, zero_division=0
        ),
        "Accuracy @ 0.5": accuracy_score(
            eligible_y_val, eligible_val_pred_05
        )
    })

eligible_validation_results = pd.DataFrame(
    eligible_validation_rows
).sort_values(
    "Validation Average Precision",
    ascending=False
).reset_index(drop=True)

print("\nEligible-discharge validation results:")
display(eligible_validation_results)


# ------------------------------------------------------------
# Select model and decision threshold using validation data only
# ------------------------------------------------------------

eligible_best_model_name = eligible_validation_results.loc[0, "Model"]
eligible_final_model = eligible_fitted_models[
    eligible_best_model_name
]

eligible_val_prob = eligible_final_model.predict_proba(
    eligible_X_val
)[:, 1]

eligible_threshold_grid = np.arange(0.05, 0.96, 0.01)
eligible_threshold_rows = []

for threshold in eligible_threshold_grid:
    eligible_val_pred = (
        eligible_val_prob >= threshold
    ).astype("int8")

    eligible_threshold_rows.append({
        "threshold": threshold,
        "precision": precision_score(
            eligible_y_val, eligible_val_pred, zero_division=0
        ),
        "recall": recall_score(
            eligible_y_val, eligible_val_pred, zero_division=0
        ),
        "f1": f1_score(
            eligible_y_val, eligible_val_pred, zero_division=0
        )
    })

eligible_threshold_results = pd.DataFrame(
    eligible_threshold_rows
)

eligible_best_row = eligible_threshold_results.loc[
    eligible_threshold_results["f1"].idxmax()
]

eligible_best_threshold = float(
    eligible_best_row["threshold"]
)

print("\nSelected eligible-cohort model:", eligible_best_model_name)
print(
    "Threshold selected on eligible validation data:",
    eligible_best_threshold
)
display(eligible_best_row.to_frame(name="Selected validation result"))


# ------------------------------------------------------------
# Final eligible-cohort test evaluation
# ------------------------------------------------------------

eligible_test_prob = eligible_final_model.predict_proba(
    eligible_X_test
)[:, 1]

eligible_test_pred = (
    eligible_test_prob >= eligible_best_threshold
).astype("int8")

eligible_cm = confusion_matrix(
    eligible_y_test,
    eligible_test_pred,
    labels=[0, 1]
)

print("\nEligible-discharge test confusion matrix:")
print(eligible_cm)

print("\nEligible-discharge classification report:")
print(classification_report(
    eligible_y_test,
    eligible_test_pred,
    labels=[0, 1],
    zero_division=0
))

eligible_test_roc_auc = roc_auc_score(
    eligible_y_test, eligible_test_prob
)

eligible_test_average_precision = average_precision_score(
    eligible_y_test, eligible_test_prob
)

eligible_tn, eligible_fp, eligible_fn, eligible_tp = (
    eligible_cm.ravel()
)

eligible_specificity = (
    eligible_tn / (eligible_tn + eligible_fp)
    if (eligible_tn + eligible_fp)
    else np.nan
)

eligible_test_precision = precision_score(
    eligible_y_test, eligible_test_pred, zero_division=0
)
eligible_test_recall = recall_score(
    eligible_y_test, eligible_test_pred, zero_division=0
)
eligible_test_f1 = f1_score(
    eligible_y_test, eligible_test_pred, zero_division=0
)
eligible_test_accuracy = accuracy_score(
    eligible_y_test, eligible_test_pred
)

print("\nEligible-discharge test metrics")
print("Selected model:", eligible_best_model_name)
print("Locked threshold:", eligible_best_threshold)
print("Test ROC-AUC:", round(eligible_test_roc_auc, 4))
print("Test average precision:", round(eligible_test_average_precision, 4))
print("Test precision:", round(eligible_test_precision, 4))
print("Test recall:", round(eligible_test_recall, 4))
print("Test F1:", round(eligible_test_f1, 4))
print("Test specificity:", round(eligible_specificity, 4))
print("Test accuracy:", round(eligible_test_accuracy, 4))


# ------------------------------------------------------------
# Store results and compare both cohorts
# ------------------------------------------------------------

eligible_selected_validation_ap = eligible_validation_results.loc[
    eligible_validation_results["Model"] == eligible_best_model_name,
    "Validation Average Precision"
].iloc[0]

eligible_cohort_metrics = {
    "Cohort": "Eligible-discharge cohort",
    "Rows": len(eligible_df),
    "Positive rate": float(eligible_y.mean()),
    "Selected model": eligible_best_model_name,
    "Validation average precision": float(
        eligible_selected_validation_ap
    ),
    "Threshold": float(eligible_best_threshold),
    "Test ROC-AUC": float(eligible_test_roc_auc),
    "Test average precision": float(eligible_test_average_precision),
    "Test precision": float(eligible_test_precision),
    "Test recall": float(eligible_test_recall),
    "Test F1": float(eligible_test_f1),
    "Test specificity": float(eligible_specificity),
    "Test accuracy": float(eligible_test_accuracy)
}

# Save the eligible-cohort artifacts separately.
# Do not overwrite the full-cohort research model or threshold.
joblib.dump(
    eligible_final_model,
    "../models/research_final_model_eligible_discharge.pkl"
)
joblib.dump(
    eligible_best_threshold,
    "../models/research_threshold_eligible_discharge.pkl"
)

cohort_comparison = pd.DataFrame([
    full_cohort_metrics,
    eligible_cohort_metrics
])

print("\nFull cohort versus eligible-discharge cohort:")
display(cohort_comparison.round(4))

print(
    "\nSaved eligible-discharge model and threshold separately. "
    "Both cohorts' test results were evaluated after model and "
    "threshold selection using their respective validation sets."
)


Eligible-discharge dataset shape: (99343, 28)
Eligible rows: 99343
Eligible unique patients: 69990
Eligible positive-class rate: 0.1139
Eligible predictor count: 26
Eligible cohort uses the same patient assignments as the full cohort.
eligible train         rows= 59773, patients= 42003, positive_rate=0.1156
eligible validation    rows= 19882, patients= 14000, positive_rate=0.1138
eligible test          rows= 19688, patients= 13987, positive_rate=0.1086

Eligible-discharge validation results:


,Model,Validation ROC-AUC,Validation Average Precision,Precision @ 0.5,Recall @ 0.5,F1 @ 0.5,Accuracy @ 0.5
0,Logistic Regression,0.670601,0.226124,0.185794,0.559434,0.278947,0.670808
1,Random Forest,0.658746,0.212178,0.500000,0.000442,0.000883,0.886178
2,Decision Tree,0.568213,0.154615,0.146987,0.399912,0.214964,0.667538
3,Dummy prior baseline,0.500000,0.113822,0.000000,0.000000,0.000000,0.886178



Selected eligible-cohort model: Logistic Regression
Threshold selected on eligible validation data: 0.5700000000000002


,Selected validation result
threshold,0.570000
precision,0.222867
recall,0.390190
f1,0.283695



Eligible-discharge test confusion matrix:
[[14569  2980]
 [ 1370   769]]

Eligible-discharge classification report:
              precision    recall  f1-score   support

           0       0.91      0.83      0.87     17549
           1       0.21      0.36      0.26      2139

    accuracy                           0.78     19688
   macro avg       0.56      0.59      0.57     19688
weighted avg       0.84      0.78      0.80     19688


Eligible-discharge test metrics
Selected model: Logistic Regression
Locked threshold: 0.5700000000000002
Test ROC-AUC: 0.6585
Test average precision: 0.2032
Test precision: 0.2051
Test recall: 0.3595
Test F1: 0.2612
Test specificity: 0.8302
Test accuracy: 0.7791

Full cohort versus eligible-discharge cohort:


,Cohort,Rows,Positive rate,Selected model,Validation average precision,Threshold,Test ROC-AUC,Test average precision,Test precision,Test recall,Test F1,Test specificity,Test accuracy
0,Full cohort,101766,0.1116,Logistic Regression,0.2253,0.56,0.6641,0.2025,0.1970,0.3919,0.2622,0.8091,0.7646
1,Eligible-discharge cohort,99343,0.1139,Logistic Regression,0.2261,0.57,0.6585,0.2032,0.2051,0.3595,0.2612,0.8302,0.7791



Saved eligible-discharge model and threshold separately. Both cohorts' test results were evaluated after model and threshold selection using their respective validation sets.


## Important Notes

- The preprocessing pipeline is fitted inside each candidate model's pipeline, using training data only.
- `patient_nbr` is used only for grouped splitting and is excluded from predictors.
- `payer_code` remains a categorical predictor.
- Infrequent categories are grouped by `OneHotEncoder` based on training data.
- The primary workflow does not use SMOTE.
- The test set is used only for the final evaluation after model and threshold selection.
- The saved threshold is not automatically applied by `Pipeline.predict()`. For threshold-based predictions, use `predict_proba()` and compare the positive-class probability with `research_threshold.pkl`.
- Ensure `A1Cresult` is retained as a categorical feature in the upstream Feature Engineering notebook.
- The grouped split aims to separate patients, not guarantee exact row-level 60/20/20 proportions.
